In [1]:
import pandas as pd
import numpy as np
import os

# INE - Data cleaning

Esta notebook muestra el proceso de limpieza  para cada uno de los indicadores seleccionados de INE a partir de las observaciones vistas en el proceso de inspección de datos. Es importante tener en cuenta que solo se van a limpiar datos, cualquier otro proceso relacionado puramente con EDA (p.ej. detectar y gestionar outliers) no forman parte del contexto de esta notebook.

Se tomarán los datasets "raw" de la carpeta 'data_raw/' y, una vez aplicada la limpieza inicial, se guardará una primera versión transformada de cada dataset en la carpeta 'data_interim/' de donde se leerán los datasets de cada indicador limpios con el fin de transformar, combinar y obtener la versión final de los datasets necesarios para pasos posteriores de este proyecto, los cuales se guardarán en 'data_clean/'.

La versión del dataset guardada en 'data_interim/' será en formato 'parquet' dado que permite guardar los datos con los cambios de tipo de datos aplicados, conservando la nueva estructura mejor que en un CSV.

### Niveles geográficos y código de concelho

Cada registro de INE está identificado por 'geocod', un código jerárquico de la nomenclatura NUTS 2024 cuya longitud indica el nivel geográfico:

| geocod | Nivel | Ejemplo |
|---|---|---|
| 'PT' | País | Portugal |
| 1 carácter | NUTS I | '1' Continente |
| 2 caracteres | NUTS II | '11' Norte |
| 3 caracteres | NUTS III | '11C' Tâmega e Sousa |
| 7 caracteres | Concelho | '11C1804' Cinfães |

En los concelhos, los 7 caracteres son **NUTS III (3) + código de concelho DICO (4)**: '11C1804' = NUTS III '11C' + concelho '1804'. Ese código de 4 dígitos es el mismo que usa E-REDES, por lo que no hace falta cruzar por nombre.

Se crea una función que añade tres columnas:
- 'nivel_geo': nivel geográfico de cada registro.
- 'cod_nuts_iii': código NUTS III del concelho.
- 'cod_concelho': código de concelho, con el mismo header común que en los datasets de E-REDES.

Las dos últimas solo se rellenan en los registros de concelho. Los registros agregados (país y regiones NUTS) se conservan en 'data_interim/': aportan los nombres oficiales de cada región NUTS, que se usan para construir la tabla de referencia geográfica, y podrán servir en el EDA a nivel regional.

In [2]:
def add_geo_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add the geographic level, NUTS III code and concelho code derived from INE's 'geocod'.

    Args:
        df (pd.DataFrame): INE dataset with a 'geocod' column (string).

    Returns:
        pd.DataFrame: The same dataset with 'nivel_geo', 'cod_nuts_iii' and 'cod_concelho' columns.
    """
    geo_levels = {1: 'nuts_i', 2: 'nuts_ii', 3: 'nuts_iii', 7: 'concelho'} # geocod length -> geographic level
    df['nivel_geo'] = df['geocod'].str.len().map(geo_levels)
    df.loc[df['geocod'] == 'PT', 'nivel_geo'] = 'pais' # 'PT' has 2 characters but is the country, not a NUTS II region
    df['nivel_geo'] = df['nivel_geo'].astype('category')

    concelho_geocod = df['geocod'].where(df['nivel_geo'] == 'concelho').astype('string') # NA for non-concelho rows
    df['cod_nuts_iii'] = concelho_geocod.str[:3]
    df['cod_concelho'] = concelho_geocod.str[-4:] # Header común para el código de concelho en todos los datasets
    return df

## **income_inhabitant_0012670**: Gross reported  income per inhabitant (€) by Geographic localization (NUTS - 2024)

In [3]:
income_inhab_df = pd.read_csv(r"../data_raw/income_inhabitant.csv", encoding='utf-8')
income_inhab_df.head(5)

,geocod,geodsg,income
0,11C1804,Cinfães,7696
1,11C1813,Resende,7839
2,1C10211,Odemira,7958
3,11B1712,Valpaços,8100
4,11D1819,Tabuaço,8191


In [4]:
income_inhab_df['geocod'] = income_inhab_df['geocod'].astype('str')
income_inhab_df['geodsg'] = income_inhab_df['geodsg'].astype('str')
income_inhab_df['income'] = income_inhab_df['income'].astype('int64')

income_inhab_df.dtypes

geocod    object
geodsg    object
income     int64
dtype: object

Se añaden el nivel geográfico, el código NUTS III y el código de concelho a partir de 'geocod', y se comprueba que hay 308 concelhos con código único.

In [5]:
income_inhab_df = add_geo_columns(income_inhab_df)
income_inhab_df['nivel_geo'].value_counts(), income_inhab_df['cod_concelho'].nunique()

(nivel_geo
 concelho    308
 nuts_iii     26
 nuts_ii       9
 nuts_i        3
 pais          1
 Name: count, dtype: int64,
 308)

In [6]:
interim_path = r"../data_interim"

os.makedirs(interim_path, exist_ok=True)

income_inhab_df.to_parquet(r"../data_interim/income_inhab.parquet")

## **pop_density_0013189**: Population's density (No./ km²) by Place of residence (NUTS - 2024)

In [7]:
pop_density_df = pd.read_csv(r"../data_raw/pop_density.csv", encoding='utf-8')
pop_density_df.head(5)

,geocod,geodsg,density
0,1500802,Alcoutim,4.3
1,1C20209,Mértola,4.6
2,1950505,Idanha-a-Nova,5.9
3,1C31203,Avis,6.0
4,1C31211,Monforte,6.8


In [8]:
pop_density_df['geocod'] = pop_density_df['geocod'].astype('str')
pop_density_df['geodsg'] = pop_density_df['geodsg'].astype('str')
pop_density_df['density'] = pop_density_df['density'].astype('float')

pop_density_df.dtypes

geocod      object
geodsg      object
density    float64
dtype: object

Igual que en el dataset anterior, se añaden las columnas geográficas a partir de 'geocod'.

In [9]:
pop_density_df = add_geo_columns(pop_density_df)
pop_density_df['nivel_geo'].value_counts(), pop_density_df['cod_concelho'].nunique()

(nivel_geo
 concelho    308
 nuts_iii     26
 nuts_ii       9
 nuts_i        3
 pais          1
 Name: count, dtype: int64,
 308)

In [10]:
pop_density_df.to_parquet(r"../data_interim/pop_density.parquet")